# 02 — RAG Pipeline


Q&A over Terms of Service and Privacy Policy documents.
Given a plain English question about any app, the system retrieves the most
relevant clause and generates a grounded answer with a citation.


## Setup: Install Dependencies

In [10]:
!pip install -q langchain-community langchain-openai langchain langchain-text-splitters python-dotenv sentence-transformers chromadb
print("✓ Dependencies installed")


✓ Dependencies installed


## Step 0: Configure API Key

In [11]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_API_KEY").startswith("sk-"):
    print("✓ OpenAI API key configured")
else:
    raise ValueError("Missing OpenAI API key")


✓ OpenAI API key configured


## Step 1: Imports

In [12]:
from typing import List
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import json
import os

print("✓ All imports successful")


✓ All imports successful


## Step 2: Load Documents

Load scraped ToS/Privacy documents from the corpus directory.


In [13]:
CORPUS_DIR = "tos_corpus"

with open(f"{CORPUS_DIR}/manifest.json") as f:
    manifest = json.load(f)

documents = []
for doc in manifest:
    if not os.path.exists(doc["filepath"]):
        continue
    with open(doc["filepath"], encoding="utf-8") as f:
        text = f.read()
    documents.append({
        "text":     text,
        "company":  doc["company"],
        "doc_type": doc["doc_type"],
        "source":   doc["filepath"],
    })

print(f"✓ Loaded {len(documents)} documents")
for d in documents:
    print(f"  {d['company']:15s} | {d['doc_type']:8s} | {len(d['text']):,} chars")


✓ Loaded 25 documents
  spotify         | tos      | 54,254 chars
  spotify         | privacy  | 39,102 chars
  tiktok          | tos      | 41,558 chars
  tiktok          | privacy  | 27,650 chars
  airbnb          | tos      | 255,682 chars
  airbnb          | privacy  | 1,632 chars
  netflix         | tos      | 56,237 chars
  netflix         | privacy  | 69,870 chars
  twitter_x       | tos      | 58,844 chars
  twitter_x       | privacy  | 33,347 chars
  google          | tos      | 28,244 chars
  google          | privacy  | 64,193 chars
  paypal          | tos      | 158,439 chars
  paypal          | privacy  | 117,212 chars
  discord         | tos      | 59,562 chars
  discord         | privacy  | 35,521 chars
  snapchat        | tos      | 98,842 chars
  snapchat        | privacy  | 32,875 chars
  youtube         | tos      | 23,990 chars
  youtube         | privacy  | 64,193 chars
  robinhood       | tos      | 3,405 chars
  tinder          | tos      | 100,317 chars
  tinder

## Step 3: Chunk Documents

Split documents into manageable chunks with overlap to preserve context.
We also store metadata (company, doc_type) alongside each chunk for filtering.


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2048,
    chunk_overlap=256,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []
metadatas = []

for doc in documents:
    doc_chunks = text_splitter.split_text(doc["text"])
    for chunk in doc_chunks:
        chunks.append(chunk)
        metadatas.append({
            "company":  doc["company"],
            "doc_type": doc["doc_type"],
            "source":   doc["source"],
        })

print(f"✓ Created {len(chunks)} chunks")
print(f"  Sample: {chunks[0][:100]}...")


✓ Created 903 chunks
  Sample: Terms and Conditions of Use - Spotify
Spotify Terms of Use
Last Updated: August 26, 2025
Introductio...


## Step 4: Embedding Model

Convert text to vectors so we can compute semantic similarity.
Using `all-MiniLM-L6-v2` — same model as class discussion notebook.



In [15]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

test_embedding = embedding_model.embed_query("What is a Terms of Service?")
print("✓ Embedding model loaded: text-embedding-3-small")
print(f"  Vector dimension: {len(test_embedding)}")



✓ Embedding model loaded: text-embedding-3-small
  Vector dimension: 1536


## Step 5: Vector Database

Index all chunks for fast similarity search.


In [16]:
vector_db = Chroma(
    collection_name="tos_documents",
    persist_directory="chroma_db",
    embedding_function=embedding_model,
)

print("✓ Existing vector database loaded")
print(f"  Collection size: {vector_db._collection.count()} chunks")



✓ Existing vector database loaded
  Collection size: 903 chunks


## Step 6: Initialize LLM

In [17]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
    max_tokens=500
)

print("✓ LLM initialized")


✓ LLM initialized


## Step 7: Retrieval Function

Retrieve the top-k most relevant chunks for a query.
Filters by company so we only retrieve from the right documents.


In [18]:
def retrieve_chunks(query: str, company: str, k: int = 3) -> List[str]:
    """Retrieve top-k most relevant chunks for a query, filtered by company."""
    results = vector_db.similarity_search(
        query,
        k=k,
        filter={"company": company}
    )
    return [doc.page_content for doc in results]


# Test it
test_chunks = retrieve_chunks("Can Spotify sell my data?", company="spotify", k=2)
print(f"Retrieved {len(test_chunks)} chunks")
print(f"  Sample: {test_chunks[0][:150]}...")


Retrieved 2 chunks
  Sample: Other Spotify users
User Data
Usage Data
Message Data
To share information about your use of the Spotify Service with other Spotify users. These could...


## Step 8: RAG Query Function

Complete RAG pipeline: retrieve relevant chunks → build prompt → generate answer.


In [19]:
def rag_query(question: str, company: str, k: int = 3) -> dict:
    """Complete RAG pipeline — retrieve then generate."""
    try:
        retrieved_chunks = retrieve_chunks(question, company=company, k=k)
        context = "\n\n".join(retrieved_chunks)

        prompt = f"""You are a legal analyst helping users understand Terms of Service and Privacy Policy documents.
Answer the question based ONLY on the provided document excerpts.
Be direct and specific. If the answer is not in the excerpts, say so.
Cite the exact clause that supports your answer.

Document excerpts:
{context}

Question: {question}

Answer:"""

        answer = llm.invoke(prompt).content

        return {
            "question": question,
            "company":  company,
            "chunks":   retrieved_chunks,
            "answer":   answer,
        }

    except Exception as e:
        print(f"Error: {e}")
        return {"error": str(e)}


# Test it
result = rag_query("Can Spotify sell my personal data to third parties?", company="spotify")
print(f"Question: {result['question']}")
print(f"\nAnswer: {result['answer']}")


Question: Can Spotify sell my personal data to third parties?

Answer: The excerpts do not explicitly state that Spotify sells personal data to third parties. They mention that personal data may be disclosed to third parties for specific purposes, such as providing services, processing payments, and delivering advertising, but there is no indication of selling personal data. 

The relevant clause states: "We will only disclose the following personal data... where you have chosen to use a Spotify Service feature, or a third party application, service or device, and we need to disclose personal data to enable this..." 

This implies sharing rather than selling.


## Step 9: Baseline Function

Vanilla LLM with no retrieval for comparison.


In [22]:
def baseline_query(question: str, company: str) -> str:
    """Vanilla LLM with no retrieval."""
    prompt = f"""You are a legal analyst. Answer this question about {company}'s Terms of Service or Privacy Policy.
Be direct and specific. If you are not sure, say so.

Question: {question}

Answer:"""
    return llm.invoke(prompt).content


# Test side by side
question = "How many days does Discord give you to opt out of arbitration?"
company  = "discord"

print("=" * 70)
print("RAG ANSWER (grounded in actual document)")
print("=" * 70)
result = rag_query(question, company=company)
print(result["answer"])

print("\n" + "=" * 70)
print("BASELINE ANSWER (no retrieval)")
print("=" * 70)
print(baseline_query(question, company=company))


RAG ANSWER (grounded in actual document)
Discord gives you 30 days to opt out of arbitration. This is supported by the clause: "You can decline this Agreement to Arbitrate... by emailing an opt-out notice... within 30 days of September 29, 2025 or when you first register your Discord account, whichever is later."

BASELINE ANSWER (no retrieval)
Discord gives you 30 days to opt out of arbitration.


## Step 10: More Test Queries

In [21]:
tests = [
    ("What is the minimum age to use Tinder?", "tinder"),
    ("What is the maximum amount X will pay in a lawsuit?", "twitter_x"),
    ("How many days does PayPal give personal account holders before changes that reduce rights?", "paypal"),
    ("What is the minimum age to use LinkedIn?", "linkedin"),
]

for question, company in tests:
    print("=" * 70)
    print(f"COMPANY:  {company.upper()}")
    print(f"QUESTION: {question}")
    print("=" * 70)
    result = rag_query(question, company=company)
    print(f"RAG: {result['answer'][:300]}")
    print(f"\nBASELINE: {baseline_query(question, company)[:300]}")
    print()


COMPANY:  TINDER
QUESTION: What is the minimum age to use Tinder?
RAG: The minimum age to use Tinder is 18 years old. This is supported by the clause: "Our service is restricted to individuals who are 18 years of age or older."

BASELINE: The minimum age to use Tinder is 18 years old.

COMPANY:  TWITTER_X
QUESTION: What is the maximum amount X will pay in a lawsuit?
RAG: The maximum amount X will pay in a lawsuit is the greater of one hundred U.S. dollars (U.S. $100.00) or the amount you paid X in the past six months for the services giving rise to the claim. This is supported by the clause: "IN NO EVENT SHALL THE AGGREGATE LIABILITY OF THE X ENTITIES EXCEED THE GRE

BASELINE: I'm sorry, but I cannot provide specific details about the maximum amount X (Twitter) will pay in a lawsuit as this information is not typically disclosed in their Terms of Service or Privacy Policy. Legal liabilities can vary widely based on the nature of the lawsuit, jurisdiction, and other factor

COMPANY:  PA